# Notebook for reading result CSV files

In [11]:
import os
import pandas as pd
from collections import defaultdict

TOP_FOLDER = "results_hashed/bucket_evaluation"
INCLUDED_FOLDERS = {"loose", "original"}  # Only check inside these, and the root

all_csv_files = []

# 1. Recursively find CSV files that contain "disk" and are inside bucket_evaluation or subfolders "loose" or "original"
for root, dirs, files in os.walk(TOP_FOLDER):
    # Check if we're in bucket_evaluation, loose, or original
    relative_root = os.path.relpath(root, TOP_FOLDER)
    top_level_dir = relative_root.split(os.sep)[0] if relative_root != "." else ""
    if top_level_dir not in INCLUDED_FOLDERS and relative_root != ".":
        continue

    for filename in files:
        if filename.lower().endswith(".csv") and "disk" in filename.lower():
            full_path = os.path.join(root, filename)
            all_csv_files.append(full_path)

# 2. Group CSVs by their *relative* folder under TOP_FOLDER
csv_by_folder = defaultdict(list)
for csv_path in all_csv_files:
    relative_path = os.path.relpath(csv_path, TOP_FOLDER)
    folder_path = os.path.dirname(relative_path)
    csv_by_folder[folder_path].append(csv_path)

# 3. Print the CSV files neatly by folder, enumerating them so we can pick by number
print(f"Found the following CSV files under '{TOP_FOLDER}' (containing 'disk' in the filename):\n")

file_index_map = {}  # Maps an integer index => full file path
index_counter = 0

for folder, files in csv_by_folder.items():
    print(f"[Folder] {folder or '(top)'}")
    for f in files:
        print(f"  {index_counter}: {os.path.basename(f)}")
        file_index_map[index_counter] = f
        index_counter += 1
    print()


Found the following CSV files under 'results_hashed/bucket_evaluation' (containing 'disk' in the filename):

[Folder] loose
  0: porto_dtw_disk_500.csv
  1: rome_frechet_disk_300_0.2-1.4_IKKE HELT FERDIG.csv
  2: rome_dtw_disk_500.csv

[Folder] loose/underveis
  3: porto_frechet_disk_300_0.2-1.2_ikke_ferdig.csv
  4: rome_frechet_300_disk_0.2-1.4_loose_IKKE HELT FERDIG.csv

[Folder] original
  5: rome_frechet_disk_300_original.csv
  6: rome_dtw_disk_500_original.csv
  7: porto_dtw_disk_500_original.csv
  8: porto_frechet_disk_300_original.csv



In [12]:
# 4. Ask user which file to open
choice = input("Enter the number of the CSV file you want to open: ")

# Basic validation in case user types something invalid
try:
    choice_index = int(choice)
    if 0 <= choice_index < len(all_csv_files):
        # 5. Read and display the chosen CSV
        chosen_file = all_csv_files[choice_index]
        print(f"\nOpening: {chosen_file}\n")
        df = pd.read_csv(chosen_file)
        display(df)  # or just print(df) if you like
    else:
        print("Invalid choice index.")
except ValueError:
    print("Please enter an integer number for your choice.")


Opening: results_hashed/bucket_evaluation/original/rome_frechet_disk_300_original.csv



,City,Measure,Resolution,Layers,Size,Threshold,Avg Precision,Avg Recall,Avg F1 Score,Avg Total Buckets,Avg Largest Bucket Size,Avg Smallest Bucket Size,Avg Buckets with >1 Trajectory,Avg Buckets with 1 Trajectory,Avg Percentage >1 Trajectory,Avg Percentage 1 Trajectory,Avg Correlation Coefficient
rome,frechet,0.2,1,10,300,0.002,0.000,0.000,0.000,8.75,15.167,1.0,4.875,3.875,58.039,41.961,NaN
rome,frechet,0.2,1,10,300,0.004,0.003,0.253,0.005,8.75,15.167,1.0,4.875,3.875,58.039,41.961,NaN
rome,frechet,0.2,1,10,300,0.006,0.009,0.301,0.018,8.75,15.167,1.0,4.875,3.875,58.039,41.961,NaN
rome,frechet,0.2,1,10,300,0.008,0.033,0.233,0.055,8.75,15.167,1.0,4.875,3.875,58.039,41.961,NaN
rome,frechet,0.2,1,10,300,0.010,0.081,0.212,0.109,8.75,15.167,1.0,4.875,3.875,58.039,41.961,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
rome,frechet,4.0,6,90,300,0.002,0.000,0.000,0.000,1800.00,1.000,1.0,0.000,1800.000,0.000,100.000,NaN
rome,frechet,4.0,6,90,300,0.004,0.000,0.000,0.000,1800.00,1.000,1.0,0.000,1800.000,0.000,100.000,NaN
rome,frechet,4.0,6,90,300,0.006,0.000,0.000,0.000,1800.00,1.000,1.0,0.000,1800.000,0.000,100.000,NaN
rome,frechet,4.0,6,90,300,0.008,0.000,0.000,0.000,1800.00,1.000,1.0,0.000,1800.000,0.000,100.000,NaN


# Visualisering av Disk Scheme resultater

In [13]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


# Get a sorted list of all Resolution values
diameters = sorted(df["Diameter"].unique())
# For demonstration, assume "Size" is the same for all rows or you want just the first row's Size
size = df["Size"].iloc[0]

for dia in diameters:
    # 1) Subset the DataFrame for this Resolution
    df_dia = df[df["Diameter"] == dia]
    
    # --- 1) Melt the dataframe ---
    df_dia_melted = df_dia.melt(
        id_vars=["Layers", "Disks", "Threshold"],  # keep these columns
        value_vars=["Avg Precision", "Avg Recall", "Avg F1 Score", "Avg Correlation Coefficient"], 
        var_name="metric",
        value_name="value"
    )

    # --- 2) Gather the unique layers and disks ---
    unique_layers = sorted(df_dia_melted["Layers"].unique())
    unique_disks = sorted(df_dia_melted["Disks"].unique())

    nrows = len(unique_disks)
    ncols = len(unique_layers)

    # --- 3) Create the figure with subplots ---
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(5 * ncols, 4 * nrows),   # adjust as needed
        sharey=False
    )

    # If there's only 1 row or 1 column, axes won't be a 2D array. Let's fix that:
    if nrows == 1 and ncols == 1:
        # Only one subplot total
        axes = [[axes]]
    elif nrows == 1:
        # axes is 1D, shape (ncols,)
        axes = [axes]
    elif ncols == 1:
        # axes is 1D, shape (nrows,)
        axes = [[ax] for ax in axes]
        
    # We’ll collect legend handles & labels from the last plot
    handles, labels = None, None

    # --- 4) Loop over each layer & disks combination ---
    for col_idx, layer_val in enumerate(unique_layers):
        for row_idx, disk_val in enumerate(unique_disks):
            ax = axes[row_idx][col_idx]
            
            # Filter for this layer & disk
            subset = df_dia_melted[
                (df_dia_melted["Layers"] == layer_val) &
                (df_dia_melted["Disks"] == disk_val)
            ]
             
            # Plot
            sns.lineplot(
                data=subset,
                x="Threshold",
                y="value",
                hue="metric",
                ax=ax
            )
            
            # -- More y‐axis ticks. Example: ticks every 0.1 from 0 to 1
            ax.yaxis.set_major_locator(MultipleLocator(0.1))
            # You can also set a fixed limit:  ax.set_ylim([0,1])
            
            ax.legend(
                loc="lower center", 
                bbox_to_anchor=(0.5, -0.6),
                ncol=2, 
                title="Metric"
            )
            
            ax.set_title(f"Diameter={format(dia, ".2f")}, Disks={disk_val},  Layers={layer_val}, (Size={size})")
            ax.set_xlabel("Threshold")
            ax.set_ylabel("Value")
            
            # Grab the handles & labels once, then remove the legend from this subplot
            handles, labels = ax.get_legend_handles_labels()
            ax.legend().remove()

    
    # Now add ONE combined legend at the bottom
    # handles/labels come from the last subplot
    if handles and labels:
        # Adjust bottom space so the legend is visible
        fig.subplots_adjust(bottom=0.25)
        
        # Add a single, centered legend below all subplots
        fig.legend(
            handles, labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.1),   # x=0.5 => center horizontally, y=0.02 => just above bottom
            ncol=2,
            title="Metric"
        )

    # Make it neat
    plt.tight_layout()
    plt.show()




KeyError: 'Diameter'

# Helping with values 

In [ ]:
# # Read in csv file from rome-dtw-100
# df = pd.read_csv("results_true/similarity_values/rome/frechet/rome-frechet-300.csv", index_col=0)

# import pandas as pd
# import numpy as np


# # Convert all values to numeric, handling errors
# df = df.apply(pd.to_numeric, errors='coerce')

# # Define the target value and tolerance
# target_value = 0.001
# tolerance = 1e-3  # Adjust if necessary

# # Find intersections close to the target value
# matches = np.isclose(df.values, target_value, atol=tolerance)
# row_indices, col_indices = np.where(matches)

# #Find lowest value that is not zero
# min_value = df[df > 0].min().min()

# print(f"Lowest value that is not zero: {min_value}")

# # Get intersection details: row name, column name, and value
# intersections = [
#     (df.index[row], df.columns[col], df.iloc[row, col])
#     for row, col in zip(row_indices, col_indices)
# ]

# # Display intersection results with values
# if intersections:
#     for row_name, col_name, value in intersections:
#         print(f"Match at intersection of '{row_name}' and '{col_name}' with value: {value}")
# else:
#     print("No match found.")


